In [36]:
%run langchain_common_notebook.py

from __future__ import annotations

from datetime import datetime
from uuid import uuid4
import pprint

import psycopg
from langchain_core.documents import Document
from langchain_classic.indexes import SQLRecordManager, index
from langchain_postgres import PGVector

In [37]:
def make_vectorstore(collection_name: str) -> PGVector:
    return PGVector(
        embeddings=databricks_embeddings,
        collection_name=collection_name,
        connection=pgvectordb_conn,
        use_jsonb=True,
    )


def make_record_manager(namespace: str) -> SQLRecordManager:
    record_manager = SQLRecordManager(namespace, db_url=pgvectordb_conn)
    record_manager.create_schema()
    return record_manager


def print_record_state(record_manager: SQLRecordManager, label: str) -> None:
    all_keys = record_manager.list_keys()
    source_a_keys = record_manager.list_keys(group_ids=["source_a"])
    source_b_keys = record_manager.list_keys(group_ids=["source_b"])

    print(f"\n{label}")
    print(f"tracked_keys_total={len(all_keys)}")
    print(f"tracked_keys_source_a={len(source_a_keys)}")
    print(f"tracked_keys_source_b={len(source_b_keys)}")

    sample = all_keys[:3]
    if sample:
        print("sample_keys:")
        for key in sample:
            print(f"  - {key[:24]}...")

In [38]:
def initial_docs() -> list[Document]:
    # Two sources are present initially: source_a and source_b.
    return [
        Document(
            page_content="Course policy v1: PA is due on every sunday at 11:55 PM.",
            metadata={"source": "source_a", "doc_id": "a1"},
        ),
        Document(
            page_content="Late work v1: you have total of 3 late days for the semester to use without penalty.",
            metadata={"source": "source_a", "doc_id": "a2"},
        ),
        Document(
            page_content="Office hours v1: Monday-Wednesday 11-1 PM.",
            metadata={"source": "source_b", "doc_id": "b1"},
        ),
    ]


def updated_subset_docs() -> list[Document]:
    # We only re-index source_a here; source_b is intentionally omitted.
    return [
        Document(
            page_content="Course policy v2: PA is now due on Fridays at 5 PM.",
            metadata={"source": "source_a", "doc_id": "a1"},
        )
    ]



In [39]:

def run_demo(cleanup_mode: str | None) -> dict:
    collection_name = "recordmgr_demo"
    namespace = f"recordmgr_demo/{collection_name}"
    
    vectorstore = make_vectorstore(collection_name)
    record_manager = make_record_manager(namespace)

    print(f"collection={collection_name}")
    print(f"namespace={namespace}")
    print(f"cleanup_mode={cleanup_mode}")

    result_initial = index(
        initial_docs(),
        record_manager,
        vectorstore,
        cleanup=None,
        source_id_key="source",
    )
    print("\nIndex pass #1 (baseline)")
    pprint.pprint(result_initial)
    print_record_state(record_manager, "RecordManager after pass #1")

    result_update = index(
        updated_subset_docs(),
        record_manager,
        vectorstore,
        cleanup=cleanup_mode,
        source_id_key="source",
    )
    print("\nIndex pass #2 (update subset)")
    pprint.pprint(result_update)
    print_record_state(record_manager, "RecordManager after pass #2")

    return {
        "collection": collection_name,
        "namespace": namespace,
        "initial": result_initial,
        "update": result_update,
    }

## Run the Three Update Modes



Each run creates a fresh collection + namespace so outputs are isolated and easy to compare.



Expected behavior in pass #2:



- `cleanup=None`: stale chunks remain

- `cleanup="incremental"`: stale chunks are removed only for `source_a`; `source_b` remains

- `cleanup="full"`: only currently indexed chunks remain


In [40]:
# cleanup=None means no stale-record deletion is performed.
# The updated source_a chunk is added, but old source_a chunks are not removed.
# source_b is not re-indexed in pass #2, and its existing chunks remain as-is.
demo_none = run_demo(cleanup_mode=None)

collection=recordmgr_demo
namespace=recordmgr_demo/recordmgr_demo
cleanup_mode=None

Index pass #1 (baseline)
{'num_added': 3, 'num_deleted': 0, 'num_skipped': 0, 'num_updated': 0}

RecordManager after pass #1
tracked_keys_total=3
tracked_keys_source_a=2
tracked_keys_source_b=1
sample_keys:
  - 6f73a882-ddfb-583a-b382-...
  - 5279a623-9377-5198-962d-...
  - b24b8ee4-5082-5e9b-980e-...

Index pass #2 (update subset)
{'num_added': 1, 'num_deleted': 0, 'num_skipped': 0, 'num_updated': 0}

RecordManager after pass #2
tracked_keys_total=4
tracked_keys_source_a=3
tracked_keys_source_b=1
sample_keys:
  - 6f73a882-ddfb-583a-b382-...
  - 5279a623-9377-5198-962d-...
  - b24b8ee4-5082-5e9b-980e-...


In [41]:
# Incremental cleanup removes stale records only for sources present in this indexing pass.
# Since this pass re-indexes only `source_a`, outdated `source_a` chunks are deleted/replaced.
# `source_b` is not included in this pass, so its existing chunks remain untouched.
demo_incremental = run_demo(cleanup_mode="incremental")

collection=recordmgr_demo
namespace=recordmgr_demo/recordmgr_demo
cleanup_mode=incremental

Index pass #1 (baseline)
{'num_added': 0, 'num_deleted': 0, 'num_skipped': 3, 'num_updated': 0}

RecordManager after pass #1
tracked_keys_total=4
tracked_keys_source_a=3
tracked_keys_source_b=1
sample_keys:
  - 80e8738a-4e52-53b1-9f4c-...
  - 6f73a882-ddfb-583a-b382-...
  - 5279a623-9377-5198-962d-...

Index pass #2 (update subset)
{'num_added': 0, 'num_deleted': 2, 'num_skipped': 1, 'num_updated': 0}

RecordManager after pass #2
tracked_keys_total=2
tracked_keys_source_a=1
tracked_keys_source_b=1
sample_keys:
  - b24b8ee4-5082-5e9b-980e-...
  - 80e8738a-4e52-53b1-9f4c-...


In [42]:
demo_full = run_demo(cleanup_mode="full")

collection=recordmgr_demo
namespace=recordmgr_demo/recordmgr_demo
cleanup_mode=full

Index pass #1 (baseline)
{'num_added': 2, 'num_deleted': 0, 'num_skipped': 1, 'num_updated': 0}

RecordManager after pass #1
tracked_keys_total=4
tracked_keys_source_a=3
tracked_keys_source_b=1
sample_keys:
  - 80e8738a-4e52-53b1-9f4c-...
  - 6f73a882-ddfb-583a-b382-...
  - 5279a623-9377-5198-962d-...

Index pass #2 (update subset)
{'num_added': 0, 'num_deleted': 3, 'num_skipped': 1, 'num_updated': 0}

RecordManager after pass #2
tracked_keys_total=1
tracked_keys_source_a=1
tracked_keys_source_b=0
sample_keys:
  - 80e8738a-4e52-53b1-9f4c-...
